In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/CSE720/code')
import os, json, time, random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import numpy as np
import pandas as pd
from PIL import Image
from torchvision.utils import save_image
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.metrics import classification_report, confusion_matrix

from config import Config
from dataset import MedicalDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = Config()
print(f"Device: {device}")

Mounted at /content/drive
Device: cuda


In [ ]:
# ============================================================
# CHANGE THIS EACH DAY
# ============================================================
STAGE = 'real_only'   # Day A: 'real_only'   |   Day B: 'augmented'
# ============================================================

In [ ]:
# ============================================================
# CHANGE THIS EACH DAY
# ============================================================
STAGE = 'augmented'   # Day A: 'real_only'   |   Day B: 'augmented'
# ============================================================

In [ ]:
def train_classifier(train_loader, val_loader, num_epochs=30, tag='real_only'):
    # Using Pretrained DenseNet121 Backbone
    model = densenet121(weights=DenseNet121_Weights.DEFAULT)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, cfg.num_domains)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    print(f"\n--- Starting Training ({tag}) with DenseNet121 ---")
    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct / len(val_loader.dataset)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            os.makedirs(cfg.classifier_dir, exist_ok=True)
            save_path = os.path.join(cfg.classifier_dir, f'{tag}_best.pth')
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
            print(f"[{tag}] epoch {epoch+1}/{num_epochs} loss={epoch_train_loss:.4f} val_acc={epoch_val_acc:.4f}")

    return model, history, best_val_acc

## Stage `real_only`

In [ ]:
if STAGE == 'real_only':
    train_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'train')
    val_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'val')
    test_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'test')

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    model, history, best_val_acc = train_classifier(train_loader, val_loader, num_epochs=30, tag='real_only')

    model.load_state_dict(torch.load(os.path.join(cfg.classifier_dir, 'real_only_best.pth')))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            preds = model(imgs.to(device)).argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())

    report = classification_report(
        all_labels, all_preds,
        target_names=[cfg.class_names[i] for i in range(cfg.num_domains)],
        output_dict=True
    )
    print(classification_report(
        all_labels, all_preds,
        target_names=[cfg.class_names[i] for i in range(cfg.num_domains)]
    ))

    with open(os.path.join(cfg.classifier_dir, 'real_only_results.json'), 'w') as f:
        json.dump({
            'test_accuracy': float(np.mean(np.array(all_preds) == np.array(all_labels))),
            'classification_report': report,
            'history': history
        }, f, indent=2)

    print("Saved real_only_results.json — now set STAGE='augmented' and re-run.")
else:
    print("STAGE is not 'real_only' — skipping this cell.")

[train] domains found: {'Diabetic Retinopathy': 400, 'Glaucoma': 400, 'Healthy': 400, 'Macular Scar': 400, 'Myopia': 400}  (total 2000)
[val] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
[test] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 159MB/s]



--- Starting Training (real_only) with DenseNet121 ---
[real_only] epoch 5/30 loss=0.5667 val_acc=0.7000
[real_only] epoch 10/30 loss=0.3688 val_acc=0.7480
[real_only] epoch 15/30 loss=0.2878 val_acc=0.6960
[real_only] epoch 20/30 loss=0.2051 val_acc=0.7080
[real_only] epoch 25/30 loss=0.2111 val_acc=0.7000
[real_only] epoch 30/30 loss=0.1952 val_acc=0.7000
                      precision    recall  f1-score   support

Diabetic_Retinopathy       0.98      0.80      0.88        50
            Glaucoma       0.50      0.30      0.38        50
             Healthy       0.56      0.88      0.68        50
        Macular_Scar       0.82      1.00      0.90        50
              Myopia       0.82      0.64      0.72        50

            accuracy                           0.72       250
           macro avg       0.73      0.72      0.71       250
        weighted avg       0.73      0.72      0.71       250

Saved real_only_results.json — now set STAGE='augmented' and re-run.


## Stage `augmented`
Generates synthetic images with your trained EyeGAN generator to expand the training set, then trains an identical classifier from scratch and evaluates on the **same** real test set as Day A.

In [ ]:
def train_classifier(train_loader, val_loader, num_epochs=30, tag='augmented'):
    # DenseNet121 Architecture Setup
    model = densenet121(weights=DenseNet121_Weights.DEFAULT)
    num_ftrs = model.classifier.in_features
    model.classifier = nn.Linear(num_ftrs, cfg.num_domains)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    print(f"\n--- Starting Training ({tag}) with DenseNet121 ---")
    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)

        epoch_train_loss = running_loss / len(train_loader.dataset)

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = correct / len(val_loader.dataset)

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            os.makedirs(cfg.classifier_dir, exist_ok=True)
            save_path = os.path.join(cfg.classifier_dir, f'{tag}_best.pth')
            torch.save(model.state_dict(), save_path)

        if (epoch + 1) % 5 == 0 or epoch == num_epochs - 1:
            print(f"[{tag}] epoch {epoch+1}/{num_epochs} loss={epoch_train_loss:.4f} val_acc={epoch_val_acc:.4f}")

    return model, history, best_val_acc

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from model import Generator
import torch.serialization
torch.serialization.add_safe_globals([Config])

# 1. Load EyeGAN Generator Model
ckpt = torch.load(os.path.join(cfg.checkpoint_dir, 'final_model.pth'), map_location=device, weights_only=False)
G = Generator(cfg.img_size, num_domains=cfg.num_domains).to(device)
G.load_state_dict(ckpt['G_state_dict'])
G.eval()

# 2. Load Real Datasets
train_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'train')
val_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'val')
test_ds = MedicalDataset(cfg.dataset_path, cfg.img_size, 'test')

# 3. Generate Synthetic Images
N_SYNTH_PER_CLASS = 50
synth_dir = os.path.join(cfg.classifier_dir, 'synthetic_images')
os.makedirs(synth_dir, exist_ok=True)

synth_paths, synth_labels = [], []
with torch.no_grad():
    for target_class in range(cfg.num_domains):
        source_idxs = random.sample(range(len(train_ds)), min(N_SYNTH_PER_CLASS, len(train_ds)))
        target_dir = os.path.join(synth_dir, cfg.class_names[target_class])
        os.makedirs(target_dir, exist_ok=True)

        for k, idx in enumerate(source_idxs):
            real_img, _, _ = train_ds[idx]
            real_img = real_img.unsqueeze(0).to(device)
            fake = G(real_img, torch.tensor([target_class], device=device))[0]
            fake_denorm = fake * 0.5 + 0.5
            out_path = os.path.join(target_dir, f'synth_{k}.png')
            save_image(fake_denorm.clamp(0, 1), out_path)


            if os.path.exists(out_path):
                synth_paths.append(out_path)
                synth_labels.append(target_class)

print(f"Generated {len(synth_paths)} synthetic images across {cfg.num_domains} classes -> {synth_dir}")

# 4. Safe Synthetic Dataset Wrapper
class SynthDataset(Dataset):
    def __init__(self, paths, labels, image_size):
        import torchvision.transforms as T
        self.paths, self.labels = paths, labels
        self.transform = T.Compose([
            T.Resize((image_size, image_size)),
            T.ToTensor(),
            T.Normalize([0.5] * 3, [0.5] * 3)
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Warning: Could not read {path}, returning dummy image. Error: {e}")
            img = Image.new('RGB', (cfg.img_size, cfg.img_size))

        return self.transform(img), self.labels[idx], path

synth_ds = SynthDataset(synth_paths, synth_labels, cfg.img_size)

# 5. Combine Real and Synthetic Training Datasets
combined_train = ConcatDataset([train_ds, synth_ds])

train_loader = DataLoader(combined_train, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

print(f"Real training images: {len(train_ds)} | + Synthetic: {len(synth_ds)} | Combined Total: {len(combined_train)}")

# 6. Train DenseNet121 Classifier on Combined Dataset
model, history, best_val_acc = train_classifier(train_loader, val_loader, num_epochs=30, tag='augmented')

# 7. Evaluate on Real Test Set
model.load_state_dict(torch.load(os.path.join(cfg.classifier_dir, 'augmented_best.pth')))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels, _ in test_loader:
        preds = model(imgs.to(device)).argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

report = classification_report(
    all_labels, all_preds,
    target_names=[cfg.class_names[i] for i in range(cfg.num_domains)],
    output_dict=True
)
print(classification_report(
    all_labels, all_preds,
    target_names=[cfg.class_names[i] for i in range(cfg.num_domains)]
))

# 8. Save Metrics
with open(os.path.join(cfg.classifier_dir, 'augmented_results.json'), 'w') as f:
    json.dump({
        'test_accuracy': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'n_synthetic_added': len(synth_ds),
        'classification_report': report,
        'history': history
    }, f, indent=2)

print("Successfully executed Augmented Stage and saved results to 'augmented_results.json'.")

[train] domains found: {'Diabetic Retinopathy': 400, 'Glaucoma': 400, 'Healthy': 400, 'Macular Scar': 400, 'Myopia': 400}  (total 2000)
[val] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
[test] domains found: {'Diabetic Retinopathy': 50, 'Glaucoma': 50, 'Healthy': 50, 'Macular Scar': 50, 'Myopia': 50}  (total 250)
Generated 250 synthetic images across 5 classes -> /content/drive/MyDrive/CSE720/revision/downstream_classifier/synthetic_images
Real training images: 2000 | + Synthetic: 250 | Combined Total: 2250

--- Starting Training (augmented) with DenseNet121 ---
[augmented] epoch 5/30 loss=0.7793 val_acc=0.6920
[augmented] epoch 10/30 loss=0.4489 val_acc=0.7120
[augmented] epoch 15/30 loss=0.2899 val_acc=0.6640
[augmented] epoch 20/30 loss=0.2231 val_acc=0.7120
[augmented] epoch 25/30 loss=0.2015 val_acc=0.6800
[augmented] epoch 30/30 loss=0.1813 val_acc=0.7280
                      precision    recall  f1-sc

## Comparison table (run once both stages are done)

In [ ]:
import os
import json as _json
import pandas as pd

real_path = os.path.join(cfg.classifier_dir, 'real_only_results.json')
aug_path = os.path.join(cfg.classifier_dir, 'augmented_results.json')

if os.path.exists(real_path) and os.path.exists(aug_path):
    real_res = _json.load(open(real_path))
    aug_res = _json.load(open(aug_path))

    print(f"Real-only test accuracy:        {real_res['test_accuracy']:.4f}")
    print(f"Real+synthetic test accuracy:   {aug_res['test_accuracy']:.4f}")
    print(f"Delta:                          {aug_res['test_accuracy']-real_res['test_accuracy']:+.4f}\n")

    rows = []
    for cls in [cfg.class_names[i] for i in range(cfg.num_domains)]:
        cls_key_real = cls if cls in real_res['classification_report'] else cls.replace(' ', '_')
        cls_key_aug = cls if cls in aug_res['classification_report'] else cls.replace(' ', '_')

        rows.append({
            'class': cls,
            'real_only_f1': real_res['classification_report'][cls_key_real]['f1-score'],
            'augmented_f1': aug_res['classification_report'][cls_key_aug]['f1-score'],
        })

    per_class = pd.DataFrame(rows)
    per_class['delta_f1'] = per_class['augmented_f1'] - per_class['real_only_f1']

    print(per_class.round(4))

    csv_out_path = os.path.join(cfg.classifier_dir, 'real_vs_augmented_per_class.csv')
    per_class.to_csv(csv_out_path, index=False)
    print(f"\nSuccessfully saved summary comparison to: {csv_out_path}")
else:
    print("Run both STAGE='real_only' and STAGE='augmented' first.")

Real-only test accuracy:        0.7240
Real+synthetic test accuracy:   0.7480
Delta:                          +0.0240

                  class  real_only_f1  augmented_f1  delta_f1
0  Diabetic_Retinopathy        0.8791        0.9293    0.0502
1              Glaucoma        0.3750        0.5275    0.1525
2               Healthy        0.6822        0.6838    0.0016
3          Macular_Scar        0.9009        0.8600   -0.0409
4                Myopia        0.7191        0.7312    0.0121

Successfully saved summary comparison to: /content/drive/MyDrive/CSE720/revision/downstream_classifier/real_vs_augmented_per_class.csv
